<a href="https://colab.research.google.com/github/eeshanwaqar/UniversalCEFR/blob/dev/EuroBert_cy_all_data_DA_B1B2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import numpy as np
import pandas as pd
import os, glob, json, csv
import matplotlib.pyplot as plt
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.metrics import (
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix,
    classification_report
)
from sklearn.model_selection import StratifiedKFold

In [ ]:
df_bt = pd.read_csv("files/welsh_back_translation_high_similarity.csv",
                    usecols=["back_translated_welsh", "cefr_level"])\
         .rename(columns={"back_translated_welsh": "text"})

In [ ]:
# Add missing columns
df_bt["title"] = "Back-translated A1/A2 sample"
df_bt["lang"] = "cy"
df_bt["source_name"] = "back_translation_pipeline_a1a2"
df_bt["format"] = "text"
df_bt["category"] = "general"
df_bt["license"] = "CC-BY-SA"

In [ ]:
df_bt

,text,cefr_level,title,lang,source_name,format,category,license
0,Roedd Ma/ penbwrdd: fy nhad yn awr yn mynd i'r...,A1,Back-translated A1/A2 sample,cy,back_translation_pipeline_a1a2,text,general,CC-BY-SA
1,Ar y acw: Byddwch yn troi at y dde yma. Ewch o...,A1,Back-translated A1/A2 sample,cy,back_translation_pipeline_a1a2,text,general,CC-BY-SA
2,Ac: mae prynhawn da. Ar y tywydd yn ofnadwy! Y...,A1,Back-translated A1/A2 sample,cy,back_translation_pipeline_a1a2,text,general,CC-BY-SA
3,Ble ydych chi'n byw?,A1,Back-translated A1/A2 sample,cy,back_translation_pipeline_a1a2,text,general,CC-BY-SA
4,Beth ydych chi'n hoffi?,A1,Back-translated A1/A2 sample,cy,back_translation_pipeline_a1a2,text,general,CC-BY-SA
...,...,...,...,...,...,...,...,...
365,Dylen nhw aros yn aros.,A2,Back-translated A1/A2 sample,cy,back_translation_pipeline_a1a2,text,general,CC-BY-SA
366,Hoffwn i fynd i Affrica.,A2,Back-translated A1/A2 sample,cy,back_translation_pipeline_a1a2,text,general,CC-BY-SA
367,A hoffech chi fynd i America?,A2,Back-translated A1/A2 sample,cy,back_translation_pipeline_a1a2,text,general,CC-BY-SA
368,A fydden nhw'n mynd i'r Almaen?,A2,Back-translated A1/A2 sample,cy,back_translation_pipeline_a1a2,text,general,CC-BY-SA


In [ ]:
# Count how many samples are labeled A1 and A2
df_bt["cefr_level"].value_counts()

,count
cefr_level,
A1,243
A2,127


In [ ]:
df_bt_b1b2 = pd.read_csv("files/high_cosine_welsh_back_translation_b1_b2.csv",
                    usecols=["back_translated_welsh", "cefr_level"])\
         .rename(columns={"back_translated_welsh": "text"})

In [ ]:
# Add missing columns
df_bt_b1b2["title"] = "Back-translated B1/B2 sample"
df_bt_b1b2["lang"] = "cy"
df_bt_b1b2["source_name"] = "back_translation_pipeline_b1b2"
df_bt_b1b2["format"] = "text"
df_bt_b1b2["category"] = "general"
df_bt_b1b2["license"] = "CC-BY-SA"

In [ ]:
df_bt_b1b2

,text,cefr_level,title,lang,source_name,format,category,license
0,Mae fy mrawd iau yn gweithio eithriad eithriad.,B1,Back-translated B1/B2 sample,cy,back_translation_pipeline_b1b2,text,general,CC-BY-SA
1,Nid yw fy chwaer yn gweithio ar hyn o bryd.,B1,Back-translated B1/B2 sample,cy,back_translation_pipeline_b1b2,text,general,CC-BY-SA
2,Ydw i'n cymdogion gyda chi?,B1,Back-translated B1/B2 sample,cy,back_translation_pipeline_b1b2,text,general,CC-BY-SA
3,Sut mae cymdogion yn gallu helpu ei gilydd?,B1,Back-translated B1/B2 sample,cy,back_translation_pipeline_b1b2,text,general,CC-BY-SA
4,Rwyt ti wedi bod problem gyda'ch cymydog? Rhy ...,B1,Back-translated B1/B2 sample,cy,back_translation_pipeline_b1b2,text,general,CC-BY-SA
...,...,...,...,...,...,...,...,...
376,Mae hi'n dweud ei fod yn heno yn y neuadd heno.,B2,Back-translated B1/B2 sample,cy,back_translation_pipeline_b1b2,text,general,CC-BY-SA
377,Mae'n meddwl ei fod yn ychydig yn gyfrinach.,B2,Back-translated B1/B2 sample,cy,back_translation_pipeline_b1b2,text,general,CC-BY-SA
378,"Dydw i ddim yn gweithio bob dydd, rwy'n mynd i...",B2,Back-translated B1/B2 sample,cy,back_translation_pipeline_b1b2,text,general,CC-BY-SA
379,Dylech gofio'r drws bob amser.,B2,Back-translated B1/B2 sample,cy,back_translation_pipeline_b1b2,text,general,CC-BY-SA


In [ ]:
# Count how many samples are labeled B1 and B2
df_bt_b1b2["cefr_level"].value_counts()

,count
cefr_level,
B1,195
B2,186


In [ ]:
# Load Welsh CEFR dataset from HuggingFace
ds = load_dataset("UniversalCEFR/learn_welsh_cy")["train"]
df_main = ds.to_pandas()

In [ ]:
df_main

,title,lang,source_name,format,category,cefr_level,license,text
0,Uned 1 - Sgwrs 1,cy,mynediad-de-learnwelsh,dialogue-level,reference,A1,public,"A: Helô, Eryl dw i. Pwy dych chi?\nB: Bore da,..."
1,Uned 1 - Sgwrs 2,cy,mynediad-de-learnwelsh,dialogue-level,reference,A1,public,"A: O na, yr heddlu! (Stopio'r car)\nB: Hello, ..."
2,Uned 1 - Sgwrs 3,cy,mynediad-de-learnwelsh,dialogue-level,reference,A1,public,"A: Bore da. Sut dych chi?\nB: Iawn, ond wedi b..."
3,Uned 2 - Sgwrs 1,cy,mynediad-de-learnwelsh,dialogue-level,reference,A1,public,"Ceri: Noswaith dda, Eryl. Sut wyt ti?\nEryl: D..."
4,Uned 2 - Sgwrs 2,cy,mynediad-de-learnwelsh,dialogue-level,reference,A1,public,A: Bore da.\nB: Hmff.\nA: Sut dych chi heddiw?...
...,...,...,...,...,...,...,...,...
1367,Uned 23 - na,cy,sylfaen-de-learnwelsh,sentence-level,reference,A2,public,Allech chi gyrraedd yn gynnar?
1368,Uned 23 - na,cy,sylfaen-de-learnwelsh,sentence-level,reference,A2,public,Allet ti gyrraedd yn gynnar?
1369,Uned 23 - na,cy,sylfaen-de-learnwelsh,sentence-level,reference,A2,public,Allai hi gyrraedd yn gynnar?
1370,Uned 23 - na,cy,sylfaen-de-learnwelsh,sentence-level,reference,A2,public,Allen nhw gyrraedd yn gynnar?


In [ ]:
df_main["cefr_level"].value_counts()

,count
cefr_level,
A1,764
A2,608


In [ ]:
# Load your B1 JSON data
df_b1 = pd.read_json("files/B1_canolradd_de-learnwelsh.json")
df_b1["cefr_level"] = "B1"
df_b1 = df_b1.drop_duplicates(subset="text", keep="first")

In [ ]:
df_b1

,title,lang,source_name,format,category,cefr_level,license,text
0,Uned 1 - Adolygu 2,cy,canolradd-de-learnwelsh,Sentence-level,reference,B1,public,Mae fy nhad i'n dod o Dreorci.
1,Uned 1 - Adolygu 3,cy,canolradd-de-learnwelsh,Sentence-level,reference,B1,public,Mae fy mrawd i'n gweithio men garej.
2,Uned 1 - Adolygu 4,cy,canolradd-de-learnwelsh,Sentence-level,reference,B1,public,Dyw fy chwaer i ddim yn gweithio ar hyn o bryd.
3,Uned 1 - Adolygu 5,cy,canolradd-de-learnwelsh,Sentence-level,reference,B1,public,Roedd fy nhad-cu i'n gweithio ar fferm.
4,Uned 1 - Siaradwch - Trafod pwnc - Cymdogion 1,cy,canolradd-de-learnwelsh,Sentence-level,reference,B1,public,Oes cymdogion da gyda chi?
...,...,...,...,...,...,...,...,...
815,Uned 14 - Lluosog - Credo gan Emyr Davies,cy,canolradd-de-learnwelsh,sentence-level,reference,B1,public,"Ar bwy, yn wir, mae’r bai?"
816,Uned 14 - Lluosog - Credo gan Emyr Davies,cy,canolradd-de-learnwelsh,sentence-level,reference,B1,public,"Mae iaith yn mynd yn ieithoedd,"
817,Uned 14 - Lluosog - Credo gan Emyr Davies,cy,canolradd-de-learnwelsh,sentence-level,reference,B1,public,Mae coeden yn troi’n goed...
818,Uned 14 - Lluosog - Credo gan Emyr Davies,cy,canolradd-de-learnwelsh,sentence-level,reference,B1,public,"Yr iaith Gymraeg, dw i’n meddwl,"


In [ ]:
df_b1["cefr_level"].value_counts()

,count
cefr_level,
B1,802


In [ ]:
# Load your B2 JSON data
df_b2 = pd.read_json("files/b2_welsh.json")
df_b2["cefr_level"] = "B2"
df_b2 = df_b2.drop_duplicates(subset="text", keep="first")

In [ ]:
df_b2

,title,lang,source_name,format,category,cefr_level,license,text
0,Uned 1 - Iaith ddefnyddiol 1,cy,uwch-1-learnwelsh,sentence-level,reference,B2,public,Mae'r ffordd yn syth iawn.
1,Uned 1 - Iaith ddefnyddiol 2,cy,uwch-1-learnwelsh,sentence-level,reference,B2,public,Mae'r oergell yn wag.
2,Uned 1 - Iaith ddefnyddiol 3,cy,uwch-1-learnwelsh,sentence-level,reference,B2,public,Mae'r stori'n wir.
3,Uned 1 - Iaith ddefnyddiol 4,cy,uwch-1-learnwelsh,sentence-level,reference,B2,public,Mae'r canlyniad yn normal.
4,Uned 1 - Iaith ddefnyddiol 5,cy,uwch-1-learnwelsh,sentence-level,reference,B2,public,Mae'r gwaith yn wahanol.
...,...,...,...,...,...,...,...,...
650,Gwaith cartref Uned 21 - Anifeiliaid anwes 8,cy,uwch-1-learnwelsh,sentence-level,reference,B2,public,"Taswn i'n gallu dal bws i'r dref, gwerthaf i f..."
651,Gwaith cartref Uned 21 - Anifeiliaid anwes 9,cy,uwch-1-learnwelsh,sentence-level,reference,B2,public,"Tasai hynny'n bosib, talodd fe am y car mewn a..."
652,Gwaith cartref Uned 21 - Anifeiliaid anwes 10,cy,uwch-1-learnwelsh,sentence-level,reference,B2,public,"Tasai waliau'r tŷ yn fwy trwchus, clywch chi m..."
653,Gwaith cartref Uned 22 - y 100 lle i'w gweld c...,cy,uwch-1-learnwelsh,document-level,reference,B2,public,Mae Llanberis yn agos at fod ar ben y rhestr o...


In [ ]:
df_b2["cefr_level"].value_counts()

,count
cefr_level,
B2,655


In [ ]:
# Combine both DataFrames
df_combined = pd.concat([df_main,df_b1,df_b2,df_bt,df_bt_b1b2], ignore_index=True)
df_combined = df_combined.drop_duplicates(subset="text", keep="first")

# Convert back to HuggingFace Dataset
ds_merged = Dataset.from_pandas(df_combined)

In [ ]:
df_combined["cefr_level"].value_counts()

,count
cefr_level,
B1,982
A1,964
B2,820
A2,719


In [ ]:
ds_merged

Dataset({
    features: ['title', 'lang', 'source_name', 'format', 'category', 'cefr_level', 'license', 'text', '__index_level_0__'],
    num_rows: 3485
})

In [ ]:
CEFR_LEVELS = ["A1", "A2","B1","B2"]
label2id = {lvl: i for i,lvl in enumerate(CEFR_LEVELS)}
labels = np.array([label2id[l] for l in ds_merged["cefr_level"]])

In [ ]:
model_name = "EuroBERT/EuroBERT-210m"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True, trust_remote_code=True)
data_collator = DataCollatorWithPadding(tokenizer)

In [ ]:
def preprocess(batch):
    toks = tokenizer(batch["text"], truncation=True, max_length=256)
    toks["labels"] = [label2id[l] for l in batch["cefr_level"]]
    return toks

In [ ]:
# Metrics
def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, labels=list(range(len(CEFR_LEVELS))), zero_division=0
    )

    metrics = {}
    for i, label in enumerate(CEFR_LEVELS):
        metrics[f"{label}_precision"] = precision[i]
        metrics[f"{label}_recall"] = recall[i]
        metrics[f"{label}_f1"] = f1[i]

    metrics["eval_accuracy"] = accuracy_score(labels, preds)
    metrics["eval_weighted_f1"] = f1_score(labels, preds, average="weighted")
    metrics["eval_weighted_precision"] = precision_score(labels, preds, average="weighted")
    metrics["eval_weighted_recall"] = recall_score(labels, preds, average="weighted")
    return metrics

In [ ]:
# Cross-validation setup
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
all_results = []

In [ ]:
import unicodedata, re, csv

# --- Text cleaning + export normalization ---
CTRL_CHARS_RE = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]")
WS_RE = re.compile(r"\s+")

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    # Normalize Welsh accents; remove control chars
    text = unicodedata.normalize("NFC", text)
    text = text.replace("\u00A0", " ")        # NBSP -> space
    return CTRL_CHARS_RE.sub("", text)

def to_one_line(text: str) -> str:
    """
    Clean + collapse all whitespace (incl. \r/\n) to a single space.
    Perfect for Excel CSVs to avoid odd spacing/row heights.
    """
    text = clean_text(text)
    text = re.sub(r"[\r\n]+", " ", text)      # kill line breaks
    text = WS_RE.sub(" ", text)               # collapse runs of whitespace
    return text.strip()


In [ ]:
# W&B auto-login and any remote telemetry
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

best_f1 = 0.0
best_trainer = None
best_tokenizer = None

all_cms = []
all_y_true, all_y_pred = [], []
all_misclassified = []

RUN_ROOT = "./eurobert_cefr_welsh_allData_b1b2_DA"
os.makedirs(RUN_ROOT, exist_ok=True)

id2label = dict(enumerate(CEFR_LEVELS))
labels_order = list(range(len(CEFR_LEVELS)))

def plot_cm(cm, labels, title, outfile, normalize=False):
    arr = cm.astype(float)
    if normalize:
        arr = arr / arr.sum(axis=1, keepdims=True).clip(min=1)
    fig, ax = plt.subplots(figsize=(6,5))
    im = ax.imshow(arr, aspect='auto')
    ax.figure.colorbar(im, ax=ax)
    ax.set(
        xticks=np.arange(len(labels)), yticks=np.arange(len(labels)),
        xticklabels=labels, yticklabels=labels,
        xlabel="Predicted", ylabel="True", title=title
    )
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    thresh = arr.max() / 2.0 if arr.size else 0
    for i in range(arr.shape[0]):
        for j in range(arr.shape[1]):
            txt = f"{arr[i,j]:.2f}" if normalize else f"{int(cm[i,j])}"
            ax.text(j, i, txt, ha="center", va="center",
                    color="white" if arr[i,j] > thresh else "black")
    fig.tight_layout(); fig.savefig(outfile, dpi=200); plt.close(fig)

for fold, (train_idx, val_idx) in enumerate(skf.split(ds_merged, labels), start=1):
    print(f"\n Running Fold {fold}...")

    ds_train = ds_merged.select(train_idx)
    ds_val = ds_merged.select(val_idx)

    tok_train = ds_train.map(preprocess, batched=True, remove_columns=ds_train.column_names)
    tok_val = ds_val.map(preprocess, batched=True, remove_columns=ds_val.column_names)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=len(CEFR_LEVELS), trust_remote_code=True
    )

    args = TrainingArguments(
        output_dir=f"{RUN_ROOT}/fold_{fold}",
        num_train_epochs=3,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=3,
        eval_strategy="epoch",
        logging_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_weighted_f1",
        greater_is_better=True,
        seed=42,
        learning_rate=3.6e-5,
        warmup_ratio=0.1,
        gradient_accumulation_steps=16,
        optim="adamw_torch_fused",
        lr_scheduler_type="linear",
        adam_beta1=0.9,
        adam_beta2=0.999,
        adam_epsilon=1e-8,
        save_total_limit=1,
        fp16=torch.cuda.is_available(),
        bf16=False
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tok_train,
        eval_dataset=tok_val,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    metrics = trainer.evaluate()

    # Track best trainer
    if metrics["eval_weighted_f1"] > best_f1:
        best_f1 = metrics["eval_weighted_f1"]
        best_trainer = trainer
        best_tokenizer = tokenizer

    # Predictions for CM + misclassified
    pred = trainer.predict(tok_val)
    logits = pred.predictions if isinstance(pred.predictions, np.ndarray) else pred.predictions[0]
    y_true = pred.label_ids
    if logits.ndim > 1:
        # confidence via softmax max (stable, simple)
        s = np.exp(logits - logits.max(axis=1, keepdims=True))
        probs = s / s.sum(axis=1, keepdims=True)
        y_pred = probs.argmax(axis=1)
        y_conf = probs.max(axis=1)
    else:
        y_pred = (logits > 0.5).astype(int)
        y_conf = np.maximum(logits, 1 - logits)

    # Per-fold CM
    cm = confusion_matrix(y_true, y_pred, labels=labels_order)
    all_cms.append(cm)
    all_y_true.append(y_true)
    all_y_pred.append(y_pred)

    # Per-fold misclassified (tidy text for Excel)
    fold_texts = ds_val["text"] if "text" in ds_val.column_names else [""] * len(y_true)
    fold_texts = [to_one_line(t) for t in fold_texts]
    fold_sources = ds_val["source_name"] if "source_name" in ds_val.column_names else ["unknown"] * len(y_true)

    df_fold = pd.DataFrame({
        "fold": fold,
        "source_name": fold_sources,
        "text": fold_texts,
        "true_label": [id2label[i] for i in y_true],
        "pred_label": [id2label[i] for i in y_pred],
        "pred_confidence": y_conf,
    })
    all_misclassified.append(df_fold[df_fold["true_label"] != df_fold["pred_label"]])

    # Store fold metrics
    row = {
        "Fold": fold,
        "All CEFR Levels Precision": metrics.get("eval_weighted_precision", 0.0),
        "All CEFR Levels Recall": metrics.get("eval_weighted_recall", 0.0),
        "All CEFR Levels F1": metrics.get("eval_weighted_f1", 0.0),
    }
    for level in CEFR_LEVELS:
        row[f"{level} Precision"] = metrics.get(f"eval_{level}_precision", 0.0)
        row[f"{level} Recall"] = metrics.get(f"eval_{level}_recall", 0.0)
        row[f"{level} F1"] = metrics.get(f"eval_{level}_f1", 0.0)
    all_results.append(row)

# --- Overall aggregation ---
all_y_true = np.concatenate(all_y_true)
all_y_pred = np.concatenate(all_y_pred)

overall_cm = confusion_matrix(all_y_true, all_y_pred, labels=labels_order)
pd.DataFrame(overall_cm, index=CEFR_LEVELS, columns=CEFR_LEVELS).to_csv(
    f"{RUN_ROOT}/confusion_matrix_overall.csv", index=True, encoding="utf-8-sig", lineterminator="\n"
)
plot_cm(overall_cm, CEFR_LEVELS, "Overall Confusion Matrix (Counts)", f"{RUN_ROOT}/cm_overall_counts.png", normalize=False)
plot_cm(overall_cm, CEFR_LEVELS, "Overall Confusion Matrix (Row-Normalized)", f"{RUN_ROOT}/cm_overall_rownorm.png", normalize=True)

# --- Misclassifications & simple post-filters/exports ---
mis_all = pd.concat(all_misclassified, ignore_index=True)
mis_all.sort_values("pred_confidence", ascending=False).to_csv(
    f"{RUN_ROOT}/misclassified.csv", index=False, encoding="utf-8-sig", lineterminator="\n"
)

# GOLD only (exclude backtranslations; handles 'back_translation_pipeline_a1a2')
if "is_backtranslation" in mis_all.columns:
    mis_gold = mis_all[~mis_all["is_backtranslation"].fillna(False)].copy()
else:
    bt_rx = r"(back[_-]?trans(?:lation)?|bt[_-]?|back_translation_pipeline)"
    meta_cols = [c for c in ["source_name","source","dataset","pipeline","origin","generator","meta_source"] if c in mis_all.columns]
    meta = mis_all[meta_cols].astype(str).agg(" ".join, axis=1) if meta_cols else pd.Series("", index=mis_all.index)
    mis_gold = mis_all[~meta.str.contains(bt_rx, case=False, na=False)].copy()

mis_gold.sort_values("pred_confidence", ascending=False).to_csv(
    f"{RUN_ROOT}/misclassified_gold.csv", index=False, encoding="utf-8-sig", lineterminator="\n"
)

# A2-only from GOLD
mis_gold_a2 = mis_gold[mis_gold["true_label"] == "A2"].copy()
mis_gold_a2.sort_values("pred_confidence", ascending=False).to_csv(
    f"{RUN_ROOT}/misclassified_gold_A2.csv", index=False, encoding="utf-8-sig", lineterminator="\n"
)

# Confidence bins → 5 groups; sample 20 each (or all if fewer)
if len(mis_gold_a2):
    mis_gold_a2["conf_bin"] = pd.qcut(
        mis_gold_a2["pred_confidence"], q=5, labels=False, duplicates="drop"
    )
    sampled_a2 = (
        mis_gold_a2.groupby("conf_bin", group_keys=False)
                   .apply(lambda g: g.sample(n=min(20, len(g)), random_state=42))
                   .reset_index(drop=True)
    )
else:
    sampled_a2 = mis_gold_a2.copy()

sampled_a2.sort_values(["conf_bin", "pred_confidence"], ascending=[True, False]).to_csv(
    f"{RUN_ROOT}/misclassified_gold_A2_sampled_by_conf.csv", index=False, encoding="utf-8-sig", lineterminator="\n"
)

print(f"\n Done. Files in {RUN_ROOT}:")
print(" - confusion_matrix_overall.csv")
print(" - cm_overall_counts.png")
print(" - cm_overall_rownorm.png")
print(" - misclassified.csv")
print(" - misclassified_gold.csv")
print(" - misclassified_gold_A2.csv")
print(" - misclassified_gold_A2_sampled_by_conf.csv")
print(f"Best eval_weighted_f1 observed: {best_f1:.4f}")


 Running Fold 1...


Map:   0%|          | 0/2788 [00:00<?, ? examples/s]

Map:   0%|          | 0/697 [00:00<?, ? examples/s]

Some weights of EuroBertForSequenceClassification were not initialized from the model checkpoint at EuroBERT/EuroBERT-210m and are newly initialized: ['classifier.bias', 'classifier.weight', 'dense.bias', 'dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/tmp/ipython-input-1814022715.py:80: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128004}.


Epoch,Training Loss,Validation Loss,Accuracy,Weighted F1,Weighted Precision,Weighted Recall,A1 Precision,A1 Recall,A1 F1,A2 Precision,A2 Recall,A2 F1,B1 Precision,B1 Recall,B1 F1,B2 Precision,B2 Recall,B2 F1
1,1.364500,1.161767,0.463415,0.451956,0.544010,0.463415,0.548673,0.321244,0.405229,0.400000,0.138889,0.206186,0.827068,0.561224,0.668693,0.326683,0.798780,0.463717
2,1.000600,1.076064,0.568149,0.545736,0.576534,0.568149,0.554717,0.761658,0.641921,0.363636,0.250000,0.296296,0.603922,0.785714,0.682927,0.756410,0.359756,0.487603
3,0.717000,0.847420,0.692970,0.693832,0.695955,0.692970,0.742574,0.777202,0.759494,0.517241,0.520833,0.519031,0.815642,0.744898,0.778667,0.654971,0.682927,0.668657


...

Map:   0%|          | 0/697 [00:00<?, ? examples/s]

Some weights of EuroBertForSequenceClassification were not initialized from the model checkpoint at EuroBERT/EuroBERT-210m and are newly initialized: ['classifier.bias', 'classifier.weight', 'dense.bias', 'dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/tmp/ipython-input-1814022715.py:80: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128004}.


Epoch,Training Loss,Validation Loss,Accuracy,Weighted F1,Weighted Precision,Weighted Recall,A1 Precision,A1 Recall,A1 F1,A2 Precision,A2 Recall,A2 F1,B1 Precision,B1 Recall,B1 F1,B2 Precision,B2 Recall,B2 F1
1,1.132400,1.000435,0.533716,0.532111,0.680162,0.533716,0.692308,0.281250,0.400000,0.325459,0.861111,0.472381,0.815029,0.715736,0.762162,0.815385,0.323171,0.462882
2,0.694000,0.766977,0.718795,0.710193,0.749955,0.718795,0.880000,0.687500,0.771930,0.821918,0.416667,0.552995,0.646617,0.873096,0.742981,0.658654,0.835366,0.736559
3,0.288800,0.580174,0.812052,0.813502,0.817401,0.812052,0.870968,0.843750,0.857143,0.700599,0.812500,0.752412,0.885870,0.827411,0.855643,0.775000,0.756098,0.765432



 Done. Files in ./eurobert_cefr_welsh_allData_b1b2_DA:
 - confusion_matrix_overall.csv
 - cm_overall_counts.png
 - cm_overall_rownorm.png
 - misclassified.csv
 - misclassified_gold.csv
 - misclassified_gold_A2.csv
 - misclassified_gold_A2_sampled_by_conf.csv
Best eval_weighted_f1 observed: 0.8394


/tmp/ipython-input-1814022715.py:171: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mis_gold = mis_all[~meta.str.contains(bt_rx, case=False, na=False)].copy()
/tmp/ipython-input-1814022715.py:190: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=min(20, len(g)), random_state=42))


In [ ]:
# Save best-performing model from all folds
final_path = "./eurobert_cefr_welsh_allData_b1b2_DA/best_model"
best_trainer.save_model(final_path)
best_tokenizer.save_pretrained(final_path)
best_trainer.state.save_to_json(os.path.join(final_path, "trainer_state.json"))

In [ ]:
# Convert to DataFrame
df = pd.DataFrame(all_results)

# Compute average row
average_row = df.drop(columns=["Fold"]).mean(numeric_only=True)
average_row["Fold"] = "Average"
df = pd.concat([df, pd.DataFrame([average_row])], ignore_index=True)

# Restructure columns
columns = [("Fold", "")] + [
    ("All CEFR Levels", "Precision"), ("All CEFR Levels", "Recall"), ("All CEFR Levels", "F1"),
    ("A1", "Precision"), ("A1", "Recall"), ("A1", "F1"),
    ("A2", "Precision"), ("A2", "Recall"), ("A2", "F1"),
    ("B1", "Precision"), ("B1", "Recall"), ("B1", "F1"),
    ("B2", "Precision"), ("B2", "Recall"), ("B2", "F1"),
]


df = df[[col[0] if col[1] == "" else f"{col[0]} {col[1]}" for col in columns]]
df.columns = pd.MultiIndex.from_tuples(columns)

In [ ]:
df

Fold All CEFR Levels                            A1                      \
                 Precision    Recall        F1 Precision    Recall        F1   
0        1        0.695955  0.692970  0.693832  0.742574  0.777202  0.759494   
1        2        0.839764  0.839311  0.839373  0.898477  0.917098  0.907692   
2        3        0.742936  0.737446  0.739123  0.759615  0.818653  0.788030   
3        4        0.842668  0.836442  0.834149  0.842105  0.911917  0.875622   
4        5        0.817401  0.812052  0.813502  0.870968  0.843750  0.857143   
5  Average        0.787745  0.783644  0.783996  0.822748  0.853724  0.837596   

         A2                            B1                            B2  \
  Precision    Recall        F1 Precision    Recall        F1 Precision   
0  0.517241  0.520833  0.519031  0.815642  0.744898  0.778667  0.654971   
1  0.711409  0.736111  0.723549  0.902062  0.892857  0.897436  0.808917   
2  0.564935  0.604167  0.583893  0.882022  0.801020  0.839572  0.713376   
3  0.750000  0.860140  0.801303  0.862069  0.888325  0.875000  0.900826   
4  0.700599  0.812500  0.752412  0.885870  0.827411  0.855643  0.775000   
5  0.648837  0.706750  0.676038  0.869533  0.830902  0.849264  0.770618   

                       
     Recall        F1  
0  0.682927  0.668657  
1  0.774390  0.791277  
2  0.682927  0.697819  
3  0.664634  0.764912  
4  0.756098  0.765432  
5  0.712195  0.737620